# Homework

Complete five true/false explanations and five coding tasks. Include imports, setup, and the relevant complete implementations in each independent answer. Sign in before saving work.

## True or False

```{index} True or False
```

**1. Path objective**

BFS’s first-discovery route always minimizes total weight in a nonnegative weighted network.

- True
- False

```{dropdown} Answer
False. BFS minimizes edge count; a costly direct edge can lose to a cheap two-edge route.
```

**2. Components**

Repeated outgoing traversal partitions every directed graph into strongly connected components.

- True
- False

```{dropdown} Answer
False. Reachability can be one-way; strongly connected groups require mutual reachability and a suitable algorithm.
```

**3. Overlap**

After KMP finds a complete occurrence, falling back through the prefix table can preserve overlapping matches.

- True
- False

```{dropdown} Answer
True. The longest proper border can begin another occurrence without backing up the text index.
```

**4. Character policy**

Ordinal UTF-16 offsets always equal user-perceived character positions.

- True
- False

```{dropdown} Answer
False. Supplementary characters take two code units, and combining sequences may involve multiple units.
```

**5. Computational limits**

Seeing an exponential enumeration implementation proves every exact algorithm for its problem takes exponential time.

- True
- False

```{dropdown} Answer
False. An implementation’s upper bound is not a lower-bound proof for the problem.
```

## Coding Questions

```{index} Coding Questions
```

## 6. Cycle and Ownership Boundaries

```{index} 6. Cycle and Ownership Boundaries
```

Run BFS and DFS on A→A,B and B→A, with Remote isolated. Mutate the caller’s adjacency after constructing GraphWalk. Check reached sets, source and unreachable paths, and rejection of an unknown source.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var adjacency=new Dictionary<string,string[]>{["A"]=new[]{"A","B","B"},["B"]=new[]{"A"},["Remote"]=Array.Empty<string>()};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
var adjacency=new Dictionary<string,string[]>{["A"]=new[]{"A","B","B"},["B"]=new[]{"A"},["Remote"]=Array.Empty<string>()};
var graph=new GraphWalk(adjacency);adjacency["A"]=Array.Empty<string>();
var bfs=graph.Bfs("A");var dfs=graph.Dfs("A");
if(bfs.Order.Length!=2||dfs.Preorder.Length!=2||bfs.PathTo("Remote").Length!=0)throw new Exception("Traversal boundary failed.");
Console.WriteLine($"BFS={string.Join(",",bfs.Order)}, DFS={string.Join(",",dfs.Preorder)}, source path={string.Join(",",bfs.PathTo("A"))}, arcs={bfs.EdgesExamined}");
try{graph.Bfs("Missing");}catch(KeyNotFoundException){Console.WriteLine("unknown source rejected");}
public sealed class GraphWalk
{
    private readonly Dictionary<string,string[]> graph;
    private readonly string[] vertices;
    public GraphWalk(Dictionary<string,string[]> adjacency)
    {
        ArgumentNullException.ThrowIfNull(adjacency);
        graph=new Dictionary<string,string[]>(StringComparer.Ordinal);
        foreach(var pair in adjacency)
        {
            if(pair.Value is null)throw new ArgumentException("Null adjacency list.");
            graph.Add(pair.Key,pair.Value.Distinct(StringComparer.Ordinal).ToArray());
        }
        foreach(var neighbors in graph.Values)
            if(neighbors.Any(v=>v is null||!graph.ContainsKey(v)))throw new ArgumentException("Unknown endpoint.");
        vertices=graph.Keys.OrderBy(v=>v,StringComparer.Ordinal).ToArray();
    }
    public sealed class Layers
    {
        public string Source { get; }
        public string[] Order { get; }
        public IReadOnlyDictionary<string,int?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        public long EdgesExamined { get; }
        internal Layers(string source,string[] order,Dictionary<string,int?> distance,Dictionary<string,string?> previous,long edges)
        {
            Source=source;Order=order;EdgesExamined=edges;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,int?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            while(at is not null)
            {
                if(path.Count>=Distance.Count)throw new InvalidOperationException("Predecessor cycle.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Invalid source path.");
            return path.ToArray();
        }
    }
    public Layers Bfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var distance=vertices.ToDictionary(v=>v,_=>(int?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new Queue<string>();var order=new List<string>();long edges=0;
        distance[source]=0;queue.Enqueue(source);
        while(queue.Count>0)
        {
            string at=queue.Dequeue();order.Add(at);trace?.Invoke($"visit {at}: layer={distance[at]}");
            foreach(string next in graph[at])
            {
                edges++;
                if(distance[next] is not null)continue;
                distance[next]=distance[at]!.Value+1;previous[next]=at;queue.Enqueue(next);
            }
        }
        return new Layers(source,order.ToArray(),distance,previous,edges);
    }
    public record Depth(string[] Preorder,string[] FinishOrder,long EdgesExamined);
    public Depth Dfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var color=vertices.ToDictionary(v=>v,_=>0,StringComparer.Ordinal);
        var stack=new Stack<(string Vertex,int Next)>();var pre=new List<string>();var finish=new List<string>();long edges=0;
        Enter(source);
        while(stack.Count>0)
        {
            var frame=stack.Pop();
            if(frame.Next==graph[frame.Vertex].Length)
            {color[frame.Vertex]=2;finish.Add(frame.Vertex);trace?.Invoke($"exit {frame.Vertex}");continue;}
            stack.Push((frame.Vertex,frame.Next+1));string next=graph[frame.Vertex][frame.Next];edges++;
            if(color[next]==0)Enter(next);
            else if(color[next]==1)trace?.Invoke($"active edge {frame.Vertex}->{next}");
        }
        return new Depth(pre.ToArray(),finish.ToArray(),edges);
        void Enter(string vertex){color[vertex]=1;pre.Add(vertex);stack.Push((vertex,0));trace?.Invoke($"enter {vertex}");}
    }
    public string[][] UndirectedComponents()
    {
        var neighbors=graph.ToDictionary(p=>p.Key,p=>p.Value.ToHashSet(StringComparer.Ordinal),StringComparer.Ordinal);
        foreach(var pair in graph)
            foreach(string next in pair.Value)
                if(!neighbors[next].Contains(pair.Key))throw new ArgumentException("Components require symmetric adjacency.");
        var seen=new HashSet<string>(StringComparer.Ordinal);var groups=new List<string[]>();
        foreach(string root in vertices)
        {
            if(!seen.Add(root))continue;
            var queue=new Queue<string>();var group=new List<string>();queue.Enqueue(root);
            while(queue.Count>0)
            {
                string at=queue.Dequeue();group.Add(at);
                foreach(string next in graph[at])if(seen.Add(next))queue.Enqueue(next);
            }
            groups.Add(group.ToArray());
        }
        return groups.ToArray();
    }
}


BFS=A,B, DFS=A,B, source path=A, arcs=3
unknown source rejected


## 7. Component Preconditions

```{index} 7. Component Preconditions
```

Count components in two undirected pairs and an isolate, then reject asymmetric directed adjacency and check the empty graph. Explain why isolates are components.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var adjacency=new Dictionary<string,string[]>{["A"]=new[]{"B"},["B"]=new[]{"A"},["C"]=new[]{"D"},["D"]=new[]{"C"},["E"]=Array.Empty<string>()};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
var adjacency=new Dictionary<string,string[]>{["A"]=new[]{"B"},["B"]=new[]{"A"},["C"]=new[]{"D"},["D"]=new[]{"C"},["E"]=Array.Empty<string>()};
var groups=new GraphWalk(adjacency).UndirectedComponents();
Console.WriteLine($"components={groups.Length}, vertices={groups.Sum(g=>g.Length)}, empty={new GraphWalk(new Dictionary<string,string[]>()).UndirectedComponents().Length}");
try{new GraphWalk(new Dictionary<string,string[]>{["A"]=new[]{"B"},["B"]=Array.Empty<string>()}).UndirectedComponents();}catch(ArgumentException){Console.WriteLine("directed asymmetry rejected");}
public sealed class GraphWalk
{
    private readonly Dictionary<string,string[]> graph;
    private readonly string[] vertices;
    public GraphWalk(Dictionary<string,string[]> adjacency)
    {
        ArgumentNullException.ThrowIfNull(adjacency);
        graph=new Dictionary<string,string[]>(StringComparer.Ordinal);
        foreach(var pair in adjacency)
        {
            if(pair.Value is null)throw new ArgumentException("Null adjacency list.");
            graph.Add(pair.Key,pair.Value.Distinct(StringComparer.Ordinal).ToArray());
        }
        foreach(var neighbors in graph.Values)
            if(neighbors.Any(v=>v is null||!graph.ContainsKey(v)))throw new ArgumentException("Unknown endpoint.");
        vertices=graph.Keys.OrderBy(v=>v,StringComparer.Ordinal).ToArray();
    }
    public sealed class Layers
    {
        public string Source { get; }
        public string[] Order { get; }
        public IReadOnlyDictionary<string,int?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        public long EdgesExamined { get; }
        internal Layers(string source,string[] order,Dictionary<string,int?> distance,Dictionary<string,string?> previous,long edges)
        {
            Source=source;Order=order;EdgesExamined=edges;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,int?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            while(at is not null)
            {
                if(path.Count>=Distance.Count)throw new InvalidOperationException("Predecessor cycle.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Invalid source path.");
            return path.ToArray();
        }
    }
    public Layers Bfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var distance=vertices.ToDictionary(v=>v,_=>(int?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new Queue<string>();var order=new List<string>();long edges=0;
        distance[source]=0;queue.Enqueue(source);
        while(queue.Count>0)
        {
            string at=queue.Dequeue();order.Add(at);trace?.Invoke($"visit {at}: layer={distance[at]}");
            foreach(string next in graph[at])
            {
                edges++;
                if(distance[next] is not null)continue;
                distance[next]=distance[at]!.Value+1;previous[next]=at;queue.Enqueue(next);
            }
        }
        return new Layers(source,order.ToArray(),distance,previous,edges);
    }
    public record Depth(string[] Preorder,string[] FinishOrder,long EdgesExamined);
    public Depth Dfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var color=vertices.ToDictionary(v=>v,_=>0,StringComparer.Ordinal);
        var stack=new Stack<(string Vertex,int Next)>();var pre=new List<string>();var finish=new List<string>();long edges=0;
        Enter(source);
        while(stack.Count>0)
        {
            var frame=stack.Pop();
            if(frame.Next==graph[frame.Vertex].Length)
            {color[frame.Vertex]=2;finish.Add(frame.Vertex);trace?.Invoke($"exit {frame.Vertex}");continue;}
            stack.Push((frame.Vertex,frame.Next+1));string next=graph[frame.Vertex][frame.Next];edges++;
            if(color[next]==0)Enter(next);
            else if(color[next]==1)trace?.Invoke($"active edge {frame.Vertex}->{next}");
        }
        return new Depth(pre.ToArray(),finish.ToArray(),edges);
        void Enter(string vertex){color[vertex]=1;pre.Add(vertex);stack.Push((vertex,0));trace?.Invoke($"enter {vertex}");}
    }
    public string[][] UndirectedComponents()
    {
        var neighbors=graph.ToDictionary(p=>p.Key,p=>p.Value.ToHashSet(StringComparer.Ordinal),StringComparer.Ordinal);
        foreach(var pair in graph)
            foreach(string next in pair.Value)
                if(!neighbors[next].Contains(pair.Key))throw new ArgumentException("Components require symmetric adjacency.");
        var seen=new HashSet<string>(StringComparer.Ordinal);var groups=new List<string[]>();
        foreach(string root in vertices)
        {
            if(!seen.Add(root))continue;
            var queue=new Queue<string>();var group=new List<string>();queue.Enqueue(root);
            while(queue.Count>0)
            {
                string at=queue.Dequeue();group.Add(at);
                foreach(string next in graph[at])if(seen.Add(next))queue.Enqueue(next);
            }
            groups.Add(group.ToArray());
        }
        return groups.ToArray();
    }
}


components=3, vertices=5, empty=0
directed asymmetry rejected


## 8. Empty, Overlapping, and Ordinal Matches

```{index} 8. Empty, Overlapping, and Ordinal Matches
```

Compare both matchers for text abababa and pattern aba, then empty pattern, missing uppercase ABA, and longer-than-text patterns. Verify first results with StringComparison.Ordinal and inspect the prefix table.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

string text="abababa";
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
string text="abababa";
foreach(string pattern in new[]{"aba","","ABA","abababab"})
{
 var d=PatternScan.Direct(text,pattern);var prepared=new PatternScan(pattern);var k=prepared.Search(text);
 int first=k.Positions.Length==0?-1:k.Positions[0];
 if(!d.Positions.SequenceEqual(k.Positions)||first!=text.IndexOf(pattern,StringComparison.Ordinal))throw new Exception("Pattern boundary failed.");
 Console.WriteLine($"pattern=[{pattern}], prefix=[{string.Join(",",prepared.Prefix)}], matches=[{string.Join(",",k.Positions)}]");
}
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


pattern=[aba], prefix=[0,0,1], matches=[0,2,4]
pattern=[], prefix=[], matches=[0,1,2,3,4,5,6,7]
pattern=[ABA], prefix=[0,0,1], matches=[]
pattern=[abababab], prefix=[0,0,1,2,3,4,5,6], matches=[]


## 9. Complete Comparison Counts

```{index} 9. Complete Comparison Counts
```

For aaaaab and aaab count all direct and KMP character comparisons, including prefix preparation and mismatches. Check the same first occurrence and repeat the prepared search to show counters describe each request.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

string text="aaaaab",pattern="aaab";
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
string text="aaaaab",pattern="aaab";
var d=PatternScan.Direct(text,pattern);var prepared=new PatternScan(pattern);var k=prepared.Search(text);var repeat=prepared.Search(text);
if(!d.Positions.SequenceEqual(k.Positions)||k.Comparisons!=repeat.Comparisons)throw new Exception("Comparison policy failed.");
Console.WriteLine($"matches={string.Join(",",k.Positions)}, direct={d.Comparisons}, prep={prepared.PreparationComparisons}, scan={k.Comparisons}, total={prepared.PreparationComparisons+k.Comparisons}");
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


matches=2, direct=12, prep=5, scan=8, total=13


## 10. Infeasibility and Exact Growth

```{index} 10. Infeasibility and Exact Growth
```

Check zero-skill and impossible cover inputs. Use BigInteger to compare 2^40 with a one-million-candidate budget, and reject the exact search’s input limit rather than labeling an oversized instance infeasible.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

int[] incomplete={0b01};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
int[] incomplete={0b01};
Console.WriteLine($"zero exact teams={TeamCover.Exact(0,Array.Empty<int>())!.Indices.Length}, impossible={TeamCover.Exact(2,incomplete) is null}");
var g=TeamCover.Greedy(2,incomplete);Console.WriteLine($"partial coverage={g.Covered}, full={g.Covered==3}");
BigInteger candidates=BigInteger.One<<40;Console.WriteLine($"2^40={candidates}, above million={candidates>1_000_000}");
try{TeamCover.Exact(1,new int[21]);}catch(ArgumentException){Console.WriteLine("input limit rejected; optimum unknown");}
public static class TeamCover
{
    public record Plan(int[] Indices,int Covered);
    // Teams are positions; each mask identifies covered requirements, up to 20.
    public static Plan Greedy(int requirements,int[] teams)
    {
        int full=Validate(requirements,teams);int covered=0;var chosen=new List<int>();
        while(covered!=full)
        {
            int best=-1,gain=0;
            for(int i=0;i<teams.Length;i++)
            {int next=System.Numerics.BitOperations.PopCount((uint)(teams[i]&~covered));if(next>gain){best=i;gain=next;}}
            if(best<0)break;
            chosen.Add(best);covered|=teams[best];
        }
        return new Plan(chosen.ToArray(),covered);
    }
    public static Plan? Exact(int requirements,int[] teams)
    {
        int full=Validate(requirements,teams);
        if(teams.Length>20)throw new ArgumentException("Exact teaching search is capped at 20 teams.");
        int bestCount=int.MaxValue,bestMask=0;
        for(int mask=0;mask<(1<<teams.Length);mask++)
        {
            int count=System.Numerics.BitOperations.PopCount((uint)mask);
            if(count>=bestCount)continue;
            int covered=0;for(int i=0;i<teams.Length;i++)if((mask&(1<<i))!=0)covered|=teams[i];
            if(covered==full){bestCount=count;bestMask=mask;}
        }
        return bestCount==int.MaxValue?null:new Plan(Enumerable.Range(0,teams.Length).Where(i=>(bestMask&(1<<i))!=0).ToArray(),full);
    }
    private static int Validate(int requirements,int[] teams)
    {
        ArgumentNullException.ThrowIfNull(teams);
        if(requirements<0||requirements>20)throw new ArgumentOutOfRangeException(nameof(requirements));
        int full=(1<<requirements)-1;
        if(teams.Any(mask=>mask<0||(mask&~full)!=0))throw new ArgumentException("Unknown requirement bit.");
        return full;
    }
}


zero exact teams=0, impossible=True
partial coverage=1, full=False
2^40=1099511627776, above million=True
input limit rejected; optimum unknown


## Submit

```{index} Submit
```

Submit explanations, code, outputs, and your contract/cost justification through the assignment panel when available.